# Steg 1: Velge EKG-utvalget

Fra 21 799 EKG til omtrent 100 EKG per diagnoseklasse, der hvert EKG har nøyaktig én diagnose. Resultat: `utvalg.csv`.

## 1. Valgene våre

In [10]:
MIN_SIKKERHET = 50        # kardiologen må være minst 50 % sikker på diagnosen
EKG_PER_KLASSE = 100      # like store klasser gir rettferdig k-NN og silhouette
MIN_KLASSESTORRELSE = 90  # mindre klasser har for få EKG til pålitelige tall
SEED = 305                # samme tilfeldige trekk hver gang

## 2. Les tabellene rett fra PhysioNet (ikke selve EKG-signalene)

In [ ]:
import pandas as pd

PTBXL = "https://physionet.org/files/ptb-xl/1.0.3/"   # versjon 1.0.3, samme som i steg 2

ekg = pd.read_csv(PTBXL + "ptbxl_database.csv", index_col="ecg_id")   # én rad per EKG
kodebok = pd.read_csv(PTBXL + "scp_statements.csv", index_col=0)    # forklaring av hver kode

print(ekg.shape)
ekg.head()

## 3. Diagnosekodene er lagret som tekst. Gjør dem om til ordbøker: kode → sikkerhet

In [12]:
import ast

ekg["scp_codes"] = ekg["scp_codes"].apply(ast.literal_eval)
ekg["scp_codes"].head(10)

ecg_id
1     {'NORM': 100.0, 'LVOLT': 0.0, 'SR': 0.0}
2                 {'NORM': 80.0, 'SBRAD': 0.0}
3                   {'NORM': 100.0, 'SR': 0.0}
4                   {'NORM': 100.0, 'SR': 0.0}
5                   {'NORM': 100.0, 'SR': 0.0}
6                   {'NORM': 100.0, 'SR': 0.0}
7                   {'NORM': 100.0, 'SR': 0.0}
8       {'IMI': 35.0, 'ABQRS': 0.0, 'SR': 0.0}
9                   {'NORM': 100.0, 'SR': 0.0}
10                  {'NORM': 100.0, 'SR': 0.0}
Name: scp_codes, dtype: object

## 4. Bare noen av kodene er diagnoser. Resten beskriver rytme eller form

In [13]:
diagnosekoder = kodebok[kodebok["diagnostic"] == 1]
print(len(kodebok), "koder, hvorav", len(diagnosekoder), "er diagnoser")


def finn_diagnoser(koder):
    funnet = []
    for kode in koder:
        if kode in diagnosekoder.index:
            funnet.append(kode)
    return funnet


# Eksempel: EKG 8 har tre koder, men bare én diagnose
print(ekg.loc[8, "scp_codes"])
print(finn_diagnoser(ekg.loc[8, "scp_codes"]))

ekg["diagnoser"] = ekg["scp_codes"].apply(finn_diagnoser)
ekg["antall_diagnoser"] = ekg["diagnoser"].apply(len)
ekg["antall_diagnoser"].value_counts()

71 koder, hvorav 44 er diagnoser
{'IMI': 35.0, 'ABQRS': 0.0, 'SR': 0.0}
['IMI']


antall_diagnoser
1    14996
2     4229
3     1512
4      528
0      411
5      119
6        4
Name: count, dtype: int64

## 5. Behold EKG med nøyaktig én diagnose, så hver prikk får én farge

In [14]:
en_diagnose = ekg[ekg["antall_diagnoser"] == 1].copy()
print(len(en_diagnose), "EKG har én diagnose")


def forste_kode(liste):
    return liste[0]


def hent_sikkerhet(rad):
    kode = rad["kode"]
    koder = rad["scp_codes"]
    return koder[kode]


en_diagnose["kode"] = en_diagnose["diagnoser"].apply(forste_kode)                     # for eksempel "ASMI"
en_diagnose["klasse"] = en_diagnose["kode"].map(diagnosekoder["diagnostic_subclass"])  # "ASMI" -> "AMI"
en_diagnose["sikkerhet"] = en_diagnose.apply(hent_sikkerhet, axis=1)                   # axis=1: én rad om gangen

en_diagnose["klasse"].value_counts()

14996 EKG har én diagnose


klasse
NORM         9069
STTC         1340
IMI          1247
AMI           583
LVH           456
LAFB/LPFB     437
NST_          435
CLBBB         353
IRBBB         311
ISCA          184
ISCI          116
CRBBB         106
IVCD           98
WPW            71
_AVB           56
LMI            37
ISC_           36
LAO/LAE        29
RAO/RAE        18
PMI             5
SEHYP           4
ILBBB           3
RVH             2
Name: count, dtype: int64

## 6. Trekk utvalget

In [15]:
sikre = en_diagnose[en_diagnose["sikkerhet"] >= MIN_SIKKERHET]

# Stokk, og sorter så de sikreste kommer først (stable: stokkingen beholdes innen lik sikkerhet)
sikre = sikre.sample(frac=1, random_state=SEED)
sikre = sikre.sort_values("sikkerhet", ascending=False, kind="stable")

# Ett EKG per pasient, ellers kan to EKG fra samme person bli naboer og gi for høy k-NN
en_per_pasient = sikre.drop_duplicates("patient_id")

utvalg = en_per_pasient.groupby("klasse").head(EKG_PER_KLASSE)
utvalg["klasse"].value_counts()

klasse
NORM         100
NST_         100
IRBBB        100
IMI          100
STTC         100
CLBBB        100
AMI          100
LVH          100
ISCA         100
LAFB/LPFB    100
ISCI         100
CRBBB         98
IVCD          94
_AVB          52
WPW           49
ISC_          35
LAO/LAE       24
RAO/RAE       17
LMI           14
SEHYP          4
ILBBB          3
PMI            3
RVH            1
Name: count, dtype: int64

## 7. Ta bort klasser som er for små

In [16]:
per_klasse = utvalg["klasse"].value_counts()
store_klasser = per_klasse[per_klasse >= MIN_KLASSESTORRELSE].index
utvalg = utvalg[utvalg["klasse"].isin(store_klasser)]

print(len(utvalg), "EKG i", utvalg["klasse"].nunique(), "klasser")

1292 EKG i 13 klasser


## 8. Lagre. `filename_hr` sier hvor 500 Hz-signalet ligger på PhysioNet, og brukes i steg 2

In [17]:
kolonner = ["patient_id", "klasse", "kode", "sikkerhet", "strat_fold", "filename_hr"]
utvalg[kolonner].sort_index().to_csv("utvalg.csv")

pd.read_csv("utvalg.csv").head()

,ecg_id,patient_id,klasse,kode,sikkerhet,strat_fold,filename_hr
0,16,10999.0,NORM,NORM,100.0,6,records500/00000/00016_hr
1,30,19053.0,LVH,LVH,100.0,9,records500/00000/00030_hr
2,41,8420.0,LAFB/LPFB,LAFB,100.0,9,records500/00000/00041_hr
3,49,13958.0,IVCD,IVCD,100.0,9,records500/00000/00049_hr
4,96,21698.0,LVH,LVH,100.0,1,records500/00000/00096_hr
